# Recognition evaluation

Use authorized institutional captures only. No images, templates, identity mappings or executed outputs belong in version control. Run the cells against your dataset to produce evaluation results.

Prepare `data/evaluation/manifest.csv` with columns `identity,capture_session,split,path`. Every path points to an authorized local image. Split values are `enrollment`, `validation`, or `test`. An identity/capture-session pair may occur in only one split. Validation and test each need known identities from separate sessions and unenrolled identities; unknown identities must also differ between validation and test.

Add optional `lighting`, `pose`, `blur`, and `occlusion` columns to inspect capture-condition subsets. Keep original capture sessions together even when images are augmented. Do not tune on the test split.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# Run from the repository root or the notebooks directory.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
from vision.evaluation import compare, distance_distributions, extract_manifest, validate_split

manifest_path = Path("data/evaluation/manifest.csv")
if not manifest_path.exists():
    raise FileNotFoundError(
        "Prepare an authorized capture manifest at data/evaluation/manifest.csv"
    )
manifest = pd.read_csv(manifest_path, dtype=str)
validate_split(manifest)
manifest.groupby(["split", "identity"]).size().rename("samples").to_frame()

## Detection, alignment and embeddings

The extraction function uses the same detector, per-face alignment and FaceNet preprocessing as the application, rejects failed quality checks and duplicate image content, and returns 512-dimensional unit vectors. This may take time on CPU. Keep vectors only in memory or approved encrypted storage.

In [ ]:
from backend.config import get_settings

settings = get_settings()
embeddings = extract_manifest(manifest, settings.asset_dir, settings.device)
assert embeddings.shape == (len(manifest), 512)
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)
embeddings.shape

In [ ]:
distances = distance_distributions(manifest, embeddings)
px.histogram(
    distances,
    x="cosine_distance",
    color="kind",
    barmode="overlay",
    nbins=50,
    title="Within-person and between-person distances",
)

## Validation and held-out comparison

Set a false-acceptance target *before* inspecting test results. The search chooses a threshold/margin using validation only and then measures test samples. A small unknown sample count cannot demonstrate a low population false-acceptance rate: report counts and uncertainty, and collect more captures. PyNNDescent falls back to exact search below ten enrollment vectors; results explicitly state whether approximate search ran.

Index build latency includes initial ANN compilation. Per-query latency measures matching only, not face detection and extraction. Python allocation peak excludes native array/model allocations. Use an OS profiler for whole-process memory. Frame API responses report server processing time as `latency_ms`.

In [ ]:
TARGET_FALSE_ACCEPTANCE = 0.01
results = compare(manifest, embeddings, target_far=TARGET_FALSE_ACCEPTANCE)
summary = pd.DataFrame(
    [{k: v for k, v in result.items() if not k.startswith("confusion_")} for result in results]
)
summary

In [ ]:
for result in results:
    if "confusion_matrix" in result:
        figure = px.imshow(
            result["confusion_matrix"],
            x=[str(x) for x in result["confusion_labels"]],
            y=[str(x) for x in result["confusion_labels"]],
            labels={"x": "Predicted (-1 = unknown)", "y": "True"},
            title=f"{result['method']} — held-out confusion matrix",
            text_auto=True,
        )
        figure.show()

## Capture conditions and update cost

Keep selected validation thresholds fixed when comparing test lighting, pose, blur and occlusion. Collect enough captures in each condition to report interpretable sample counts. Adding a new identity below measures a full snapshot rebuild, including new templates; it does not measure image extraction or encrypted database writes.

In [ ]:
import time

from vision.evaluation import evaluate
from vision.recognition import RecognitionIndex

training = manifest.split.eq("enrollment").to_numpy()
identity_map = {
    name: index for index, name in enumerate(sorted(manifest.loc[training, "identity"].unique()))
}
labels = np.array([identity_map.get(name, -1) for name in manifest.identity])
condition_results = []
for result in results:
    if "error" in result:
        continue
    index = RecognitionIndex(embeddings[training], labels[training], method=result["method"])
    for condition in ["lighting", "pose", "blur", "occlusion"]:
        if condition not in manifest:
            continue
        for value in manifest[condition].dropna().unique():
            mask = (manifest.split.eq("test") & manifest[condition].eq(value)).to_numpy()
            if mask.any():
                measured = evaluate(
                    index, embeddings[mask], labels[mask], result["threshold"], result["margin"]
                )
                condition_results.append(
                    {
                        "method": result["method"],
                        "condition": condition,
                        "value": value,
                        **{k: v for k, v in measured.items() if not k.startswith("confusion_")},
                    }
                )
pd.DataFrame(condition_results)

In [ ]:
# Timing a rebuild using an authorized validation unknown as a new enrollment identity.
# This is a separate update-cost experiment, never input to the held-out metrics above.
unknown = manifest.split.eq("validation").to_numpy() & (labels == -1)
new_identity = manifest.loc[unknown, "identity"].iloc[0]
new_samples = (manifest.identity.eq(new_identity) & manifest.split.eq("validation")).to_numpy()
updated_vectors = np.concatenate([embeddings[training], embeddings[new_samples]])
updated_labels = np.concatenate([labels[training], np.full(new_samples.sum(), len(identity_map))])
update_timings = []
for method in ["cosine", "knn", "ann"]:
    started = time.perf_counter()
    updated_index = RecognitionIndex(updated_vectors, updated_labels, method=method)
    update_timings.append(
        {"method": method, "snapshot_rebuild_ms": (time.perf_counter() - started) * 1000}
    )
pd.DataFrame(update_timings)

## Model selection and limitations

Choose a method from measured unknown rejection, known identification errors, update cost and latency; do not pick only the highest closed-set accuracy. Re-test the chosen configuration on a new capture session before adopting it. The application uses cosine matching. Daily stations mark stable matches automatically by default; staff review is configurable. Scheduled sessions require staff verification.

StandardScaler is not used by default: centering/scaling changes FaceNet cosine geometry and needs its own train-only fit and validation experiment. A neural classifier is deferred until there are enough independent captures per identity to justify its capacity. A closed-set MLP still needs an independent open-set rejection policy.

Presentation attacks are not evaluated here. Blinking, temporal stability and similarity thresholds do not establish robust liveness. Do not use recognition alone for disciplinary, academic or payroll decisions.

Write a measured interpretation including sample counts, camera/device, capture conditions, false matches, uncertainty, limitations and chosen thresholds. Do not save notebook outputs containing personal data to Git.

In [ ]:
# Optional local aggregate report (no images, templates, or names).
output = Path("data/evaluation/results.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(
        results,
        indent=2,
        default=lambda value: value.item() if isinstance(value, np.generic) else str(value),
    )
)
output